# whisper-audit — 在浏览器里试 / Try it in your browser

中文长音频 → 文稿/字幕，带**覆盖率审计**：量实际音量找出「有人在说但没转出字」的区段，定点补转，并给你质检报告。
Chinese long-audio transcription that **audits its own output** for dropped speech.

**运行前**：菜单 `代码执行程序 → 更改运行时类型 → T4 GPU`（Runtime → Change runtime type → T4 GPU）。
全程约 5 分钟：安装 ~1 分钟、模型下载 ~2 分钟、转录 7 分钟样本 ~1 分钟。

[GitHub](https://github.com/xr843/whisper-audit) · [实测数据](https://github.com/xr843/whisper-audit/blob/master/docs/measurements.md) · [踩坑记录](https://github.com/xr843/whisper-audit/blob/master/docs/lessons.md)

In [ ]:
# 1) 确认 GPU 在位（没有也能跑，只是慢很多）
!nvidia-smi -L || echo '⚠ 未检测到 GPU——菜单里切换运行时类型后重跑本单元格'

In [ ]:
# 2) 安装（Colab 自带 torch 与 ffmpeg，所以不需要 [cuda] extra；
#    [socks] 给走 SOCKS 代理下载模型的环境，30KB 无副作用）
%pip install -q "whisper-audit[whisper,socks]"
import importlib.metadata as _m
print('whisper-audit', _m.version('whisper-audit'), '已就绪')

In [ ]:
# 3) 下载演示音频：鲁迅《阿Q正传》第一章朗读，7 分 46 秒
#    LibriVox 录音，公有领域（Public Domain Mark 1.0）
!curl -sL -o sample.mp3 https://archive.org/download/truestoryahq_1612_librivox/trueQ_01_lu_64kb.mp3
!ls -lh sample.mp3

In [ ]:
# 4) 转录。默认档 = 单路 + 覆盖率审计 + 定点补转
!whisper-audit run sample.mp3 -o out \
  && echo '✅ 转录完成' \
  || echo '❌ 转录失败——请把上面的报错开 issue：https://github.com/xr843/whisper-audit/issues'

In [ ]:
# 5) 看成稿（时间戳分段 + 自动标点）
print(open('out/sample_全文转录.txt', encoding='utf-8').read()[:1500])

In [ ]:
# 6) 质检报告——本工具的核心：告诉你转录本身可信到什么程度
import json
r = json.load(open('out/质检报告.json', encoding='utf-8'))
f = r['final']
print(f"时间覆盖率   {f['cover_pct']:.1f}%")
print(f"有效语音     {f['speech_pct']:.1f}%（按字数密度估）")
print(f"残余可疑段   {f['starved']} 处")
print(f"剔除的幻觉   {len(r['hallucinations'])} 处")
print(f"自动改动记账 {len(r['pinyin_fixes'])} 处（每处带时间戳，可回听核对）")

In [ ]:
# 7) 字幕（词级时间戳重切，配原音频逐句回听）
print(open('out/sample_字幕.srt', encoding='utf-8').read()[:600])

## 转你自己的录音 / Your own audio

跑下面的单元格上传（mp3 / m4a / wav 都行）。音频只进这台临时 Colab 机器，不经过任何第三方服务；会话结束即销毁。

In [ ]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import files
    up = files.upload()          # Run all 跑到这里会等你选文件
    if not up:
        print('未选择文件——跳过（想转自己的录音时再单独运行本格）')
    else:
        name = next(iter(up))
        !whisper-audit run "{name}" -o my_out
        import glob
        print(open(glob.glob('my_out/*_全文转录.txt')[0], encoding='utf-8').read()[:1200])
else:
    print('本单元格只在 Colab 里用（本地直接跑 CLI 即可）')

## 接下来 / Next

| 场景 | 命令 |
|---|---|
| 清晰普通话（讲课/会议）——最准且快 | `--engine funasr` |
| 多人对话标注说话人 | `--engine funasr --diarize` |
| 赶时间（62x 实时） | `--profile fast` |
| 量你这份录音的正确率 | `whisper-audit goldset` → 改错字 → `eval` |

本地安装：`pip install "whisper-audit[whisper,cuda]"`（GPU）或 `pip install "whisper-audit[funasr]"` 后加 `--device cpu`（无显卡，3.2x 实时）。

引擎怎么选、每个默认值为什么是这个数：[docs/measurements.md](https://github.com/xr843/whisper-audit/blob/master/docs/measurements.md)